In [1]:
import joblib
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [2]:
rf2017_path = "models/random_forest_cicids2017.joblib"
rf2023_path = "models/random_forest_ciciot2023.joblib"

model_cicids = joblib.load(rf2017_path)["model"]
model_ciciot = joblib.load(rf2023_path)["model"]

print("Loaded both models.")

Loaded both models.


In [4]:
df_cicids2017 = load_cicids()

Total files: 8
Reading: C:\Users\Rasmus\.cache\kagglehub\datasets\chethuhn\network-intrusion-dataset\versions\1\Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Reading: C:\Users\Rasmus\.cache\kagglehub\datasets\chethuhn\network-intrusion-dataset\versions\1\Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Reading: C:\Users\Rasmus\.cache\kagglehub\datasets\chethuhn\network-intrusion-dataset\versions\1\Friday-WorkingHours-Morning.pcap_ISCX.csv
Reading: C:\Users\Rasmus\.cache\kagglehub\datasets\chethuhn\network-intrusion-dataset\versions\1\Monday-WorkingHours.pcap_ISCX.csv
Reading: C:\Users\Rasmus\.cache\kagglehub\datasets\chethuhn\network-intrusion-dataset\versions\1\Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Reading: C:\Users\Rasmus\.cache\kagglehub\datasets\chethuhn\network-intrusion-dataset\versions\1\Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Reading: C:\Users\Rasmus\.cache\kagglehub\datasets\chethuhn\network-intrusion-dataset\versions\1\Tuesday-Worki

In [5]:
df_ciciot2023 = load_ciciot(base_path="./../data/CICIoT2023/")

Listing CSV files in: ./../data/CICIoT2023/
Total files: 6
Reading: ./../data/CICIoT2023/BenignTraffic.pcap.csv
Reading: ./../data/CICIoT2023/BenignTraffic1.pcap.csv
Reading: ./../data/CICIoT2023/BenignTraffic2.pcap.csv
Reading: ./../data/CICIoT2023/BenignTraffic3.pcap.csv
Reading: ./../data/CICIoT2023/DoS-HTTP_Flood.pcap.csv
Reading: ./../data/CICIoT2023/DoS-HTTP_Flood1.pcap.csv
Full Shape: (1170052, 40)
Before cleaning: (1170052, 40)
Removed 4623 duplicates
Dropped 4691 missing values
After cleaning: (1165361, 40)
Label
BENIGN      1093876
dos_http      71485
Name: count, dtype: int64


In [6]:
# CICIDS2017: BENIGN vs everything else
df_cicids2017["Label"] = df_cicids2017["Label"].apply(lambda x: "BENIGN" if str(x).upper() == "BENIGN" else "ATTACK")

# CICIoT2023: BENIGN vs dos_http
df_ciciot2023 = df_ciciot2023[df_ciciot2023["Label"].isin(["BENIGN", "dos_http"])].copy()
df_ciciot2023["Label"] = df_ciciot2023["Label"].replace({
    "BENIGN": "BENIGN",
    "dos_http": "ATTACK"
})

print(df_cicids2017["Label"].value_counts())
print(df_ciciot2023["Label"].value_counts())

Label
BENIGN    2095057
ATTACK     425741
Name: count, dtype: int64
Label
BENIGN    1093876
ATTACK      71485
Name: count, dtype: int64


In [7]:
target_column = "Label"

X_cicids = df_cicids2017.drop(columns=[target_column])
y_cicids = df_cicids2017[target_column]

X_ciciot = df_ciciot2023.drop(columns=[target_column])
y_ciciot = df_ciciot2023[target_column]

In [10]:
print("CICIDS test shape:", X_cicids.shape)
print("CICIoT test shape:", X_ciciot.shape)

print("Model CICIDS expects:", getattr(model_cicids, "n_features_in_", "unknown"))
print("Model CICIoT expects:", getattr(model_ciciot, "n_features_in_", "unknown"))

CICIDS test shape: (2520798, 76)
CICIoT test shape: (1165361, 39)
Model CICIDS expects: unknown
Model CICIoT expects: unknown


In [14]:
y_pred_cicids_on_ciciot = model_cicids.predict(X_ciciot)

print("=== CICIDS2017-trained model tested on CICIoT2023 ===")
print("Accuracy:", accuracy_score(y_ciciot, y_pred_cicids_on_ciciot))
print("\nClassification Report:")
print(classification_report(y_ciciot, y_pred_cicids_on_ciciot))
print("\nConfusion Matrix:")
print(confusion_matrix(y_ciciot, y_pred_cicids_on_ciciot))

ValueError: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- ARP
- AVG
- DHCP
- DNS
- HTTP
- ...
Feature names seen at fit time, yet now missing:
- ACK Flag Count
- Active Max
- Active Mean
- Active Min
- Active Std
- ...


In [12]:
print(type(model_cicids))
print(model_cicids.keys())

<class 'dict'>
dict_keys(['model'])
